In [4]:
import torch
import torch.nn as nn

In [5]:
class SelfAttention(nn.Module):
    def __init__(self,embed_size,heads):
        super(SelfAttention,self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        assert(self.head_dim * heads == embed_size), "Embed size needs to be div by heads"

        self.values = nn.Linear(self.head_dim,self.head_dim,bias=False)
        self.keys = nn.Linear(self.head_dim,self.head_dim,bias=False)
        self.queries = nn.Linear(self.head_dim,self.head_dim,bias=False)
        self.fc_out =nn.Linear(heads*self.head_dim,embed_size)
    def forward(self,values,keys,query,mask):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        #split embedding into self.heads pieces
        values = values.reshape(N,value_len,self.heads,self.head_dim)
        keys = keys.reshape(N,key_len,self.heads,self.head_dim)
        query = query.reshape(N,query_len,self.heads,self.head_dim)

        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(query)

        energy = torch.einsum("nqhd,nkhd->nhqk",[queries,keys])
        # n: batch size, q: query length, h: no. of heads, d: head dim
        # n: batch size, k: key length, h: no. of heads, d: head dim

        if mask is not None:
            energy = energy.masked_fill(mask==0,float("-1e20"))

        attention = torch.softmax(energy/(self.embed_size**(1/2)),dim=3)

        out = torch.einsum("nhql,nlhd->nqhd",[attention,values])\
            .reshape(
            N,query_len,self.heads*self.head_dim
        )
        # n: batch size, q: query length, h: no. of heads, d: head dim, l: value length

        out = self.fc_out(out)
        return out    

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self,embed_size,heads,dropout,forward_expansion):
        super(TransformerBlock,self).__init__()
        self.attention = SelfAttention(embed_size,heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size,forward_expansion*embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion*embed_size,embed_size)
        )
        self.dropout = nn.Dropout(dropout)

        
    def forward(self,value,key,query,mask):
        attention  = self.attention(value,key,query,mask)
        input = self.dropout(self.norm1(attention + query))
        forward_from_NN = self.feed_forward(input)
        output = self.dropout(self.norm2(forward_from_NN + input))
        return output


In [7]:
class Encoder(nn.Module):
    def __init__(
            self, src_vocab_size, embed_size, num_layers, heads, device, forward_expansion, dropout, max_length
    ):
        super(Encoder,self).__init__()
        self.embed_size = embed_size
        self.device = device
        self.word_embedding = nn.Embedding(src_vocab_size,embed_size)
        self.position_embedding = nn.Embedding(max_length,embed_size)

        self.layers = nn.ModuleList(
            [
                TransformerBlock(
                    embed_size,
                    heads,
                    dropout=dropout,
                    forward_expansion=forward_expansion
                )
                for _ in range(num_layers)
            ]
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self,x,mask):
        N, seq_length = x.shape
        positions = torch.arange(0,seq_length).expand(N,seq_length).to(self.device)
        out = self.dropout(self.word_embedding(x) + self.position_embedding(positions))

        for layer in self.layers:
            out = layer(out,out,out,mask)

        return out    

In [8]:
class DecoderBlock(nn.Module):
    def __init__(self,embed_size,heads,forward_expansion,dropout,device):
        super(DecoderBlock,self).__init__()
        self.attention = SelfAttention(embed_size,heads)
        self.norm = nn.LayerNorm(embed_size)
        self.transformer_block = TransformerBlock(
            embed_size,heads,dropout,forward_expansion
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self,x,value,key,src_mask,trg_mask):
        attention = self.attention(x,x,x,trg_mask)
        query = self.dropout(self.norm(attention + x))
        out = self.transformer_block(value,key,query,src_mask)
        return out

In [9]:
class Decoder(nn.Module):
    def __init__(
            self,
            trg_vocab_size,
            embed_size,
            num_layers,
            heads,
            forward_expansion,
            dropout,
            device,
            max_length
    ):
        super(Decoder,self).__init__()
        self.device = device
        self.word_embedding = nn.Embedding(trg_vocab_size,embed_size)
        self.position_embedding = nn.Embedding(max_length,embed_size)

        self.layers = nn.ModuleList(
            [
                DecoderBlock(
                    embed_size,
                    heads,
                    forward_expansion,
                    dropout,
                    device
                )
                for _ in range(num_layers)
            ]
        )
        self.fc_out = nn.Linear(embed_size,trg_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x,enc_out,src_mask,trg_mask):
        N, seq_length = x.shape
        positions = torch.arange(0,seq_length).expand(N,seq_length).to(self.device)
        x = self.dropout(self.word_embedding(x) + self.position_embedding(positions))

        for layer in self.layers:
            x = layer(x,enc_out,enc_out,src_mask,trg_mask)

        out = self.fc_out(x)

        return out

In [10]:
class Transformer(nn.Module):
    def __init__(
            self,
            src_vocab_size,
            trg_vocab_size,
            src_pad_idx,
            trg_pad_idx,
            embed_size=256,
            num_layers=6,
            forward_expansion=4,
            heads=8,
            dropout=0,
            device="cuda",
            max_length=100
    ):
        super(Transformer,self).__init__()

        self.encoder = Encoder(
            src_vocab_size,
            embed_size,
            num_layers,
            heads,
            device,
            forward_expansion,
            dropout,
            max_length
        )

        self.decoder = Decoder(
            trg_vocab_size,
            embed_size,
            num_layers,
            heads,
            forward_expansion,
            dropout,
            device,
            max_length
        )

        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

    def make_src_mask(self,src):
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        # (N,1,1,src_len)
        return src_mask.to(self.device)

    def make_trg_mask(self,trg):
        N, trg_len = trg.shape
        trg_mask = torch.tril(torch.ones((trg_len,trg_len))).expand(
            N,1,trg_len,trg_len
        )
        return trg_mask.to(self.device)

    def forward(self,src,trg):
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        enc_src = self.encoder(src,src_mask)
        out = self.decoder(trg,enc_src,src_mask,trg_mask)
        return out

In [11]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)
    x = torch.tensor([[1,5,6,4,3,9,5],[1,8,7,3,4,2,0]]).to(device)
    trg = torch.tensor([[1,7,4,3,5,9,2],[1,5,6,2,4,3,0]]).to(device)
    src_pad_idx = 0
    trg_pad_idx = 0
    src_vocab_size = 10
    trg_vocab_size = 10
    model = Transformer(
        src_vocab_size,
        trg_vocab_size,
        src_pad_idx,
        trg_pad_idx,
        device=device,
        max_length=100
    ).to(device)

    out = model(x,trg[:,:-1])
    print(out.shape) # (N,trg_len - 1 , trg_vocab_size)

cpu
torch.Size([2, 6, 10])


In [12]:
# Text preprocessing and vocabulary utilities
import re
from collections import Counter
import torch.nn.functional as F

class TextProcessor:
    def __init__(self):
        self.word_to_idx = {}
        self.idx_to_word = {}
        self.vocab_size = 0
        
    def build_vocab(self, texts):
        """Build vocabulary from training texts"""
        # Combine all texts and tokenize
        all_words = []
        for text in texts:
            # Simple tokenization - split by spaces and lowercase
            words = re.findall(r'\b\w+\b', text.lower())
            all_words.extend(words)
        
        # Count word frequencies
        word_counts = Counter(all_words)
        
        # Create vocabulary - start with special tokens
        vocab = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
        vocab.extend([word for word, count in word_counts.most_common()])
        
        # Create mappings
        self.word_to_idx = {word: idx for idx, word in enumerate(vocab)}
        self.idx_to_word = {idx: word for idx, word in enumerate(vocab)}
        self.vocab_size = len(vocab)
        
        print(f"Vocabulary size: {self.vocab_size}")

        print(f"Sample vocabulary: {vocab[:20]}")

        print(f"Sample word to index mapping: {list(self.word_to_idx.items())[:20]}")
        print(f"Sample index to word mapping: {list(self.idx_to_word.items())[:20]}")


    def text_to_indices(self, text, max_length=20):
        """Convert text to sequence of indices"""
        words = re.findall(r'\b\w+\b', text.lower())
        indices = [self.word_to_idx.get('<SOS>')]
        
        for word in words[:max_length-2]:  # Leave room for SOS and EOS
            indices.append(self.word_to_idx.get(word, self.word_to_idx.get('<UNK>')))
        
        indices.append(self.word_to_idx.get('<EOS>'))
        
        # Pad sequence
        while len(indices) < max_length:
            indices.append(self.word_to_idx.get('<PAD>'))
            
        return indices[:max_length]
    
    def indices_to_text(self, indices):
        """Convert indices back to text"""
        words = []
        for idx in indices:
            word = self.idx_to_word.get(idx, '<UNK>')
            if word == '<EOS>':
                break
            if word not in ['<PAD>', '<SOS>']:
                words.append(word)
        return ' '.join(words)

In [13]:
# Create training dataset - simple question-answer pairs
training_data = [
    ("what is the capital of france", "the capital of france is paris"),
    ("what is the capital of italy", "the capital of italy is rome"),
    ("what is the capital of spain", "the capital of spain is madrid"),
    ("what is the capital of germany", "the capital of germany is berlin"),
    ("what is the capital of england", "the capital of england is london"),
    ("what color is the sky", "the sky is blue"),
    ("what color is grass", "grass is green"),
    ("what color is snow", "snow is white"),
    ("what animal says meow", "cats say meow"),
    ("what animal says woof", "dogs say woof"),
    ("what animal says moo", "cows say moo"),
    ("how are you", "i am fine thank you"),
    ("what is your name", "i am a transformer model"),
    ("good morning", "good morning to you too"),
    ("good night", "good night sleep well"),
    ("thank you", "you are welcome"),
    ("hello there", "hello how are you"),
    ("what is two plus two", "two plus two equals four"),
    ("what is three plus three", "three plus three equals six"),
    ("what is five plus five", "five plus five equals ten"),
]

print(f"Training dataset size: {len(training_data)}")
print("Sample training pairs:")
for i, (inp, out) in enumerate(training_data[:5]):
    print(f"{i+1}. Input: '{inp}' -> Output: '{out}'")

Training dataset size: 20
Sample training pairs:
1. Input: 'what is the capital of france' -> Output: 'the capital of france is paris'
2. Input: 'what is the capital of italy' -> Output: 'the capital of italy is rome'
3. Input: 'what is the capital of spain' -> Output: 'the capital of spain is madrid'
4. Input: 'what is the capital of germany' -> Output: 'the capital of germany is berlin'
5. Input: 'what is the capital of england' -> Output: 'the capital of england is london'


In [14]:
# Preprocess the data
processor = TextProcessor()

# Extract all texts for vocabulary building
all_texts = []
for inp, out in training_data:
    all_texts.extend([inp, out])

# Build vocabulary
processor.build_vocab(all_texts)

# Convert text data to sequences
max_length = 15  # Maximum sequence length
processed_data = []

for inp, out in training_data:
    inp_seq = processor.text_to_indices(inp, max_length)
    out_seq = processor.text_to_indices(out, max_length)
    processed_data.append((inp_seq, out_seq))

print(f"\nProcessed {len(processed_data)} training pairs")
print(f"Sequence length: {max_length}")
print(f"Sample processed pair:")
inp_seq, out_seq = processed_data[0]
print(f"Input sequence: {inp_seq}")
print(f"Output sequence: {out_seq}")
print(f"Input text: '{processor.indices_to_text(inp_seq)}'")
print(f"Output text: '{processor.indices_to_text(out_seq)}'")

# Model parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

src_vocab_size = processor.vocab_size
trg_vocab_size = processor.vocab_size
src_pad_idx = processor.word_to_idx['<PAD>']
trg_pad_idx = processor.word_to_idx['<PAD>']
embed_size = 128
num_layers = 2
heads = 4
forward_expansion = 2
dropout = 0.1

print(f"Model configuration:")
print(f"- Vocabulary size: {src_vocab_size}")
print(f"- Embedding size: {embed_size}")
print(f"- Number of layers: {num_layers}")
print(f"- Number of heads: {heads}")

Vocabulary size: 65
Sample vocabulary: ['<PAD>', '<SOS>', '<EOS>', '<UNK>', 'is', 'what', 'the', 'capital', 'of', 'you', 'plus', 'good', 'two', 'three', 'five', 'color', 'animal', 'says', 'say', 'are']
Sample word to index mapping: [('<PAD>', 0), ('<SOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('is', 4), ('what', 5), ('the', 6), ('capital', 7), ('of', 8), ('you', 9), ('plus', 10), ('good', 11), ('two', 12), ('three', 13), ('five', 14), ('color', 15), ('animal', 16), ('says', 17), ('say', 18), ('are', 19)]
Sample index to word mapping: [(0, '<PAD>'), (1, '<SOS>'), (2, '<EOS>'), (3, '<UNK>'), (4, 'is'), (5, 'what'), (6, 'the'), (7, 'capital'), (8, 'of'), (9, 'you'), (10, 'plus'), (11, 'good'), (12, 'two'), (13, 'three'), (14, 'five'), (15, 'color'), (16, 'animal'), (17, 'says'), (18, 'say'), (19, 'are')]

Processed 20 training pairs
Sequence length: 15
Sample processed pair:
Input sequence: [1, 5, 4, 6, 7, 8, 21, 2, 0, 0, 0, 0, 0, 0, 0]
Output sequence: [1, 6, 7, 8, 21, 4, 39, 2, 0, 0, 0, 0, 0

In [19]:
# Create and train the model
model = Transformer(
    src_vocab_size=src_vocab_size,
    trg_vocab_size=trg_vocab_size,
    src_pad_idx=src_pad_idx,
    trg_pad_idx=trg_pad_idx,
    embed_size=embed_size,
    num_layers=num_layers,
    forward_expansion=forward_expansion,
    heads=heads,
    dropout=dropout,
    device=device,
    max_length=max_length
).to(device)

# Training setup
criterion = nn.CrossEntropyLoss(ignore_index=src_pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Convert data to tensors
train_inputs = []
train_targets = []

for inp_seq, out_seq in processed_data:
    train_inputs.append(inp_seq)
    train_targets.append(out_seq)

train_inputs = torch.tensor(train_inputs).to(device)
train_targets = torch.tensor(train_targets).to(device)

print(f"Training data shape:")
print(f"Inputs: {train_inputs.shape}")
print(f"Targets: {train_targets.shape}")

# Training loop
num_epochs = 100
print(f"\nStarting training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    # Simple batch training (using all data as one batch for simplicity)
    optimizer.zero_grad()
    
    # Forward pass
    # Use target sequence without the last token as input, predict next token
    output = model(train_inputs, train_targets[:, :-1])
    
    # Reshape for loss calculation
    output = output.reshape(-1, trg_vocab_size)
    target = train_targets[:, 1:].reshape(-1)  # Target is shifted by one position
    
    # Calculate loss
    loss = criterion(output, target)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    total_loss += loss.item()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}")

print("Training completed!")

Training data shape:
Inputs: torch.Size([20, 15])
Targets: torch.Size([20, 15])

Starting training for 100 epochs...
Epoch [20/100], Loss: 1.9827
Epoch [40/100], Loss: 0.8964
Epoch [60/100], Loss: 0.4635
Epoch [80/100], Loss: 0.2902
Epoch [100/100], Loss: 0.2015
Training completed!


In [16]:
# Inference function
def generate_response(model, processor, input_text, max_length=15, device='cpu'):
    """Generate response for given input text"""
    model.eval()
    
    # Convert input text to sequence
    input_seq = processor.text_to_indices(input_text, max_length)
    input_tensor = torch.tensor([input_seq]).to(device)
    
    # Start with SOS token for target
    sos_idx = processor.word_to_idx['<SOS>']
    eos_idx = processor.word_to_idx['<EOS>']
    
    # Generated sequence starts with SOS
    generated = [sos_idx]
    
    with torch.no_grad():
        for _ in range(max_length - 1):
            # Create target tensor from generated sequence so far
            target_tensor = torch.tensor([generated]).to(device)
            
            # Get model output
            output = model(input_tensor, target_tensor)
            
            # Get the last token's probabilities
            last_token_logits = output[0, -1, :]
            
            # Get the most likely next token
            next_token = torch.argmax(last_token_logits).item()
            
            # Add to generated sequence
            generated.append(next_token)
            
            # Stop if we generate EOS token
            if next_token == eos_idx:
                break
    
    # Convert generated sequence to text
    response = processor.indices_to_text(generated)
    return response

print("Inference function ready!")

Inference function ready!


In [17]:
# Test the trained model
print("Testing the trained transformer model:\n")

# Test cases - mix of seen and unseen inputs
test_inputs = [
    "what is the capital of france",  
    "what is the capital of italy",   
    "what color is the sky",          
    "what is the capital of japan",    
    "what color is the sun",          
    "hello there",                    
    "good morning",                   
    "what animal says meow",          
    "what is five plus five",
]

print("Model responses:")
print("-" * 60)

for i, test_input in enumerate(test_inputs):
    response = generate_response(model, processor, test_input, max_length, device)
    print(f"{i+1}. Input: '{test_input}'")
    print(f"   Output: '{response}'")
    print()

print("-" * 60)
print("Training completed! The model can now answer basic questions.")
print("Note: With such a small dataset and simple architecture,")
print("the model works best on questions similar to the training data.")

Testing the trained transformer model:

Model responses:
------------------------------------------------------------
1. Input: 'what is the capital of france'
   Output: 'the capital of france is paris'

2. Input: 'what is the capital of italy'
   Output: 'the capital of italy is rome'

3. Input: 'what color is the sky'
   Output: 'the sky is blue'

4. Input: 'what is the capital of japan'
   Output: 'the capital of spain is berlin'

5. Input: 'what color is the sun'
   Output: 'the sky is blue'

6. Input: 'hello there'
   Output: 'hello how are you'

7. Input: 'good morning'
   Output: 'good morning to you too'

8. Input: 'what animal says meow'
   Output: 'cats say meow'

9. Input: 'what is five plus five'
   Output: 'five plus five equals ten'

------------------------------------------------------------
Training completed! The model can now answer basic questions.
Note: With such a small dataset and simple architecture,
the model works best on questions similar to the training dat

In [20]:
# Interactive testing - Try your own questions!
def ask_model(question):
    """Ask the model a question and get a response"""
    response = generate_response(model, processor, question, max_length, device)
    print(f"You: {question}")
    print(f"Model: {response}")
    print("-" * 40)
print("=" * 40)

# Test some examples
ask_model("what is the capital of germany")
ask_model("what color is snow")
ask_model("what is two + two?")

You: what is the capital of germany
Model: the capital of germany is berlin
----------------------------------------
You: what color is snow
Model: snow is white
----------------------------------------
You: what is two + two?
Model: two plus two equals four
----------------------------------------
